In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision import models

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Data
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),  # AlexNet min input: 63×63; 224 is standard
        transforms.ToTensor(),
    ]
)

train_set = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
test_set = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)
train_loader = DataLoader(
    train_set, batch_size=64, shuffle=True, num_workers=4, pin_memory=True
)
test_loader = DataLoader(
    test_set, batch_size=64, shuffle=False, num_workers=4, pin_memory=True
)


100.0%
/home/popraf/Desktop/studia/D7047E_Advanced_deep_learning_assignments/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [5]:
# Helpers
def train(model, optimizer, criterion, loader, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(
            f"  Epoch [{epoch + 1:>2}/{epochs}]  Loss: {total_loss / len(loader):.4f}"
        )


def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = 100.0 * correct / total
    print(f"  ➜  Test Accuracy: {acc:.2f}%")
    return acc


criterion = nn.CrossEntropyLoss()

In [6]:
# EXPERIMENT 1 — Fine-Tuning (random init)
model_ft = models.alexnet(weights=None)  # all weights randomly initialised
model_ft.classifier[6] = nn.Linear(4096, 10)  # replace final FC: 1000 → 10
model_ft = model_ft.to(device)

optimizer_ft = optim.Adam(model_ft.parameters(), lr=1e-4)
train(model_ft, optimizer_ft, criterion, train_loader, epochs=10)
acc_ft = evaluate(model_ft, test_loader)

  Epoch [ 1/10]  Loss: 1.6428
  Epoch [ 2/10]  Loss: 1.1843
  Epoch [ 3/10]  Loss: 0.9635
  Epoch [ 4/10]  Loss: 0.8090
  Epoch [ 5/10]  Loss: 0.6945
  Epoch [ 6/10]  Loss: 0.6076
  Epoch [ 7/10]  Loss: 0.5320
  Epoch [ 8/10]  Loss: 0.4630
  Epoch [ 9/10]  Loss: 0.4019
  Epoch [10/10]  Loss: 0.3500
  ➜  Test Accuracy: 80.71%


In [7]:
# EXPERIMENT 2 — Feature Extraction (pretrained)
model_fe = models.alexnet(
    weights=models.AlexNet_Weights.IMAGENET1K_V1
)  # load pretrained

# Freeze ALL layers
for param in model_fe.parameters():
    param.requires_grad = False

# Replace & unfreeze ONLY the final FC layer
model_fe.classifier[6] = nn.Linear(4096, 10)  # new layer is trainable by default
model_fe = model_fe.to(device)

# Pass only trainable params to optimizer
optimizer_fe = optim.Adam(
    filter(lambda p: p.requires_grad, model_fe.parameters()), lr=1e-3
)
train(model_fe, optimizer_fe, criterion, train_loader, epochs=10)
acc_fe = evaluate(model_fe, test_loader)

3.0%

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /home/popraf/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100.0%


  Epoch [ 1/10]  Loss: 1.1991
  Epoch [ 2/10]  Loss: 1.0936
  Epoch [ 3/10]  Loss: 1.0674
  Epoch [ 4/10]  Loss: 1.0634
  Epoch [ 5/10]  Loss: 1.0571
  Epoch [ 6/10]  Loss: 1.0557
  Epoch [ 7/10]  Loss: 1.0478
  Epoch [ 8/10]  Loss: 1.0456
  Epoch [ 9/10]  Loss: 1.0413
  Epoch [10/10]  Loss: 1.0450
  ➜  Test Accuracy: 67.39%


In [8]:
print(f"Fine-Tuning   (random init):  {acc_ft:.2f}%")
print(f"Feature Extr. (pretrained):   {acc_fe:.2f}%")

Fine-Tuning   (random init):  80.71%
Feature Extr. (pretrained):   67.39%


##### Why Fine-Tuning (80.71%) Beats Feature Extraction (67.39%)
1. Domain Mismatch — The Core Reason

ImageNet images are high-resolution, real-world photos (224×224+). CIFAR-10 images are tiny 32×32 pixels, upscaled to 224×224 for AlexNet. The pretrained convolutional filters were trained to detect fine-grained textures and details that simply don't exist in low-resolution CIFAR-10 images. In Feature Extraction, those frozen filters produce poor feature representations for CIFAR-10, and only the tiny FC head can compensate — which it can't do enough on its own.

2. Feature Extraction Is Too Constrained

In Experiment 2, only Linear(1000->10) — roughly 10,010 parameters — are updated. The frozen backbone is essentially a mismatched feature extractor for CIFAR-10's blurry, upscaled content. Fine-Tuning trains the entire network (~60M parameters), allowing all layers to adapt to the CIFAR-10 distribution.

3. CIFAR-10 Is Large Enough to Train From Scratch

The conventional wisdom "pretrained = better" holds when the target dataset is small (e.g., a few hundred images). CIFAR-10 has 50,000 training images — enough for AlexNet to learn meaningful features from scratch without overfitting badly.


Pretrained weights are a strong prior for similar domains. When the domain differs significantly (ImageNet HD photos vs. CIFAR-10 tiny images), freezing the backbone locks in the wrong features, and full fine-tuning wins because the model has the freedom to fully adapt.